# The Complete Transformer Encoder-Decoder

This notebook assembles the components developed in notebook 3 into a complete sequence-to-sequence Transformer. It adds:

- a reusable English–German parallel-text dataset,
- dynamic per-batch padding and teacher-forcing shifts,
- stacked encoder and decoder blocks,
- shared source/target token embeddings and a tied output projection, and
- padding-aware next-token cross-entropy loss.

The implementation follows the post-layer-normalization structure of the original Transformer.

In [1]:
from itertools import islice, zip_longest
import json
import math
from pathlib import Path
import shutil
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from huggingface_hub import HfApi
from safetensors.torch import save_model
from tokenizers import Tokenizer
from tqdm.auto import tqdm

## 1. Configuration and tokenizer

The original Transformer used six encoder and six decoder layers. We retain that depth and the base model's 512-dimensional hidden states. The training section is configured for one 24 GB NVIDIA RTX 3090 Ti.

In [2]:
embedding_dim = 512
num_heads = 8
num_layers = 6
context_length = 128
dropout_rate = 0.1
label_smoothing = 0.1
batch_size = 64
max_examples = None  # Use the complete corpus for training.


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing pyproject.toml")


repo_root = find_repo_root()
tokenizer_path = repo_root / "data" / "tokenizers" / "tatoeba_bpe_en_de.json"
english_path = repo_root / "data" / "raw" / "tatoeba_en_de" / "Tatoeba.de-en.en"
german_path = repo_root / "data" / "raw" / "tatoeba_en_de" / "Tatoeba.de-en.de"

tokenizer = Tokenizer.from_file(str(tokenizer_path))
vocab_size = tokenizer.get_vocab_size()
pad_token_id = tokenizer.token_to_id("[PAD]")
bos_token_id = tokenizer.token_to_id("[BOS]")
eos_token_id = tokenizer.token_to_id("[EOS]")

assert None not in (pad_token_id, bos_token_id, eos_token_id)
print(f"Vocabulary size: {vocab_size:,}")
print(f"PAD/BOS/EOS ids: {pad_token_id}/{bos_token_id}/{eos_token_id}")

Vocabulary size: 16,000
PAD/BOS/EOS ids: 0/1/2


## 2. Parallel dataset and dynamic batching

The two Tatoeba files are line-aligned: English line `i` is paired with German line `i`. The dataset stores text pairs and tokenizes examples on demand. The collator then pads only to the longest source and target in the current batch.

Truncation retains `[EOS]`, which is important because it teaches the decoder when to stop generating.

In [3]:
class ParallelTextDataset(Dataset):
    def __init__(
        self,
        source_path,
        target_path,
        tokenizer,
        max_seq_len,
        max_examples=None,
    ):
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.eos_token_id = tokenizer.token_to_id("[EOS]")

        pairs = []
        with (
            open(source_path, encoding="utf-8") as source_file,
            open(target_path, encoding="utf-8") as target_file,
        ):
            lines = zip_longest(source_file, target_file)
            if max_examples is not None:
                lines = islice(lines, max_examples)

            for source_line, target_line in lines:
                if source_line is None or target_line is None:
                    raise ValueError(
                        "Source and target files must contain the same number of lines"
                    )
                source_text = source_line.rstrip()
                target_text = target_line.rstrip()
                if source_text and target_text:
                    pairs.append((source_text, target_text))

        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def _encode(self, text):
        token_ids = self.tokenizer.encode(text).ids
        if len(token_ids) > self.max_seq_len:
            token_ids = token_ids[: self.max_seq_len - 1] + [self.eos_token_id]
        return token_ids

    def __getitem__(self, index):
        source_text, target_text = self.pairs[index]
        return {
            "source_ids": self._encode(source_text),
            "target_ids": self._encode(target_text),
        }


class TranslationCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def _pad(self, sequences):
        max_len = max(len(sequence) for sequence in sequences)
        padded = [
            sequence + [self.pad_token_id] * (max_len - len(sequence))
            for sequence in sequences
        ]
        return torch.tensor(padded, dtype=torch.long)

    def __call__(self, examples):
        source_ids = self._pad([example["source_ids"] for example in examples])
        target_ids = self._pad([example["target_ids"] for example in examples])

        decoder_input_ids = target_ids[:, :-1]
        labels = target_ids[:, 1:]

        return {
            "source_ids": source_ids,
            "decoder_input_ids": decoder_input_ids,
            "labels": labels,
            "source_padding_mask": source_ids.eq(self.pad_token_id),
            "decoder_padding_mask": decoder_input_ids.eq(self.pad_token_id),
        }

In [4]:
all_dataset = ParallelTextDataset(
    source_path=english_path,
    target_path=german_path,
    tokenizer=tokenizer,
    max_seq_len=context_length,
    max_examples=max_examples,
)

preview_loader = DataLoader(
    all_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=TranslationCollator(pad_token_id),
)

batch = next(iter(preview_loader))
for name, tensor in batch.items():
    print(f"{name:22s} {tuple(tensor.shape)}")

source_ids             (64, 27)
decoder_input_ids      (64, 25)
labels                 (64, 25)
source_padding_mask    (64, 27)
decoder_padding_mask   (64, 25)


## 3. Components from notebook 3

These are the reusable positional encoding, multi-head attention, encoder-block, and decoder-block definitions. The standalone component tests from notebook 3 are intentionally omitted.

In [5]:
class SinusoidalPositionEmbedding(nn.Module):
    def __init__(self, context_length, embedding_dim):
        super().__init__()
        position = torch.arange(context_length).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, embedding_dim, 2)
            * (-torch.log(torch.tensor(10_000.0)) / embedding_dim)
        )

        pe = torch.zeros(context_length, embedding_dim)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, positions):
        return self.pe[positions]


class MultiheadAttention(nn.Module):
    def __init__(
        self,
        embedding_dim,
        num_heads,
        context_length,
        is_crossattention=False,
        is_causal=False,
        dropout_rate=0.1,
    ):
        super().__init__()
        assert embedding_dim % num_heads == 0
        assert not (is_crossattention and is_causal)

        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads
        self.is_crossattention = is_crossattention
        self.is_causal = is_causal

        self.w_q = nn.Linear(embedding_dim, embedding_dim)
        self.w_k = nn.Linear(embedding_dim, embedding_dim)
        self.w_v = nn.Linear(embedding_dim, embedding_dim)
        self.w_o = nn.Linear(embedding_dim, embedding_dim)
        self.attn_dropout = nn.Dropout(dropout_rate)

        if is_causal:
            causal_mask = torch.triu(
                torch.ones(context_length, context_length, dtype=torch.bool),
                diagonal=1,
            )
            self.register_buffer("causal_mask", causal_mask)

    def forward(self, hidden_states, key_value_states=None, padding_mask=None):
        batch_size, query_len, embedding_dim = hidden_states.shape
        assert embedding_dim == self.embedding_dim

        if self.is_crossattention:
            assert key_value_states is not None
        else:
            assert key_value_states is None
            key_value_states = hidden_states

        assert key_value_states.shape[0] == batch_size
        assert key_value_states.shape[2] == self.embedding_dim
        key_value_len = key_value_states.shape[1]

        q = self.w_q(hidden_states)
        k = self.w_k(key_value_states)
        v = self.w_v(key_value_states)

        q = q.view(batch_size, query_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, key_value_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, key_value_len, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = q @ k.transpose(-2, -1) / (self.head_dim**0.5)

        if self.is_causal:
            assert query_len <= self.causal_mask.shape[0]
            assert key_value_len <= self.causal_mask.shape[1]
            causal_mask = self.causal_mask[:query_len, :key_value_len]
            attn_scores = attn_scores.masked_fill(
                causal_mask[None, None, :, :], float("-inf")
            )

        if padding_mask is not None:
            assert padding_mask.shape == (batch_size, key_value_len)
            assert padding_mask.dtype == torch.bool
            attn_scores = attn_scores.masked_fill(
                padding_mask[:, None, None, :], float("-inf")
            )

        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)
        output = attn_weights @ v
        output = output.transpose(1, 2).contiguous().view(batch_size, query_len, embedding_dim)
        return self.w_o(output)

In [6]:
class EncoderBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads, context_length, dropout_rate=0.1):
        super().__init__()
        self.self_attention = MultiheadAttention(
            embedding_dim, num_heads, context_length, dropout_rate=dropout_rate
        )
        self.ffn = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.ReLU(),
            nn.Linear(4 * embedding_dim, embedding_dim),
        )
        self.attn_dropout = nn.Dropout(dropout_rate)
        self.ffn_dropout = nn.Dropout(dropout_rate)
        self.layer_norm1 = nn.LayerNorm(embedding_dim)
        self.layer_norm2 = nn.LayerNorm(embedding_dim)

    def forward(self, hidden_states, padding_mask=None):
        attn_output = self.self_attention(hidden_states, padding_mask=padding_mask)
        hidden_states = self.layer_norm1(
            hidden_states + self.attn_dropout(attn_output)
        )
        ffn_output = self.ffn(hidden_states)
        return self.layer_norm2(hidden_states + self.ffn_dropout(ffn_output))


class DecoderBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads, context_length, dropout_rate=0.1):
        super().__init__()
        self.causal_self_attention = MultiheadAttention(
            embedding_dim,
            num_heads,
            context_length,
            is_causal=True,
            dropout_rate=dropout_rate,
        )
        self.cross_attention = MultiheadAttention(
            embedding_dim,
            num_heads,
            context_length,
            is_crossattention=True,
            dropout_rate=dropout_rate,
        )
        self.ffn = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.ReLU(),
            nn.Linear(4 * embedding_dim, embedding_dim),
        )
        self.self_attn_dropout = nn.Dropout(dropout_rate)
        self.cross_attn_dropout = nn.Dropout(dropout_rate)
        self.ffn_dropout = nn.Dropout(dropout_rate)
        self.layer_norm1 = nn.LayerNorm(embedding_dim)
        self.layer_norm2 = nn.LayerNorm(embedding_dim)
        self.layer_norm3 = nn.LayerNorm(embedding_dim)

    def forward(
        self,
        hidden_states,
        encoder_hidden_states,
        decoder_padding_mask=None,
        encoder_padding_mask=None,
    ):
        attn_output = self.causal_self_attention(
            hidden_states, padding_mask=decoder_padding_mask
        )
        hidden_states = self.layer_norm1(
            hidden_states + self.self_attn_dropout(attn_output)
        )

        attn_output = self.cross_attention(
            hidden_states,
            key_value_states=encoder_hidden_states,
            padding_mask=encoder_padding_mask,
        )
        hidden_states = self.layer_norm2(
            hidden_states + self.cross_attn_dropout(attn_output)
        )

        ffn_output = self.ffn(hidden_states)
        return self.layer_norm3(hidden_states + self.ffn_dropout(ffn_output))

## 4. Stacked encoder and decoder

Every layer preserves the sequence length and embedding dimension. The encoder converts source embeddings into contextual source representations. The decoder repeatedly refines its target-side states using causal self-attention and cross-attention to the final encoder states.

In [7]:
class TransformerEncoder(nn.Module):
    def __init__(
            self,
            embedding_dim,
            num_heads,
            context_length,
            num_layers,
            dropout_rate=0.1
    ):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderBlock(
                embedding_dim=embedding_dim,
                num_heads=num_heads,
                context_length=context_length,
                dropout_rate=dropout_rate,
            )
            for _ in range(num_layers)
        ])

    def forward(self, hidden_states, padding_mask: torch.Tensor = None):
        for layer in self.layers:
            hidden_states = layer(hidden_states, padding_mask=padding_mask)

        return hidden_states
    
# testing
encoder = TransformerEncoder(512, 8, 128, 6, 0.1)
encoder

TransformerEncoder(
  (layers): ModuleList(
    (0-5): 6 x EncoderBlock(
      (self_attention): MultiheadAttention(
        (w_q): Linear(in_features=512, out_features=512, bias=True)
        (w_k): Linear(in_features=512, out_features=512, bias=True)
        (w_v): Linear(in_features=512, out_features=512, bias=True)
        (w_o): Linear(in_features=512, out_features=512, bias=True)
        (attn_dropout): Dropout(p=0.1, inplace=False)
      )
      (ffn): Sequential(
        (0): Linear(in_features=512, out_features=2048, bias=True)
        (1): ReLU()
        (2): Linear(in_features=2048, out_features=512, bias=True)
      )
      (attn_dropout): Dropout(p=0.1, inplace=False)
      (ffn_dropout): Dropout(p=0.1, inplace=False)
      (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
      (layer_norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
    )
  )
)

In [8]:
class TransformerDecoder(nn.Module):
    def __init__(
            self,
            embedding_dim,
            num_heads,
            context_length,
            num_layers,
            dropout_rate=0.1
    ):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderBlock(
                embedding_dim=embedding_dim,
                num_heads=num_heads,
                context_length=context_length,
                dropout_rate=dropout_rate,
            )
            for _ in range(num_layers)
        ])

    def forward(
            self,
            hidden_states,
            encoder_hidden_states,
            decoder_padding_mask: torch.Tensor = None,
            encoder_padding_mask: torch.Tensor = None
        ):
        for layer in self.layers:
            hidden_states = layer(
                hidden_states,
                encoder_hidden_states=encoder_hidden_states,
                decoder_padding_mask=decoder_padding_mask,
                encoder_padding_mask=encoder_padding_mask
            )

        return hidden_states
    
# testing
decoder = TransformerDecoder(512, 8, 128, 6, 0.1)
decoder

TransformerDecoder(
  (layers): ModuleList(
    (0-5): 6 x DecoderBlock(
      (causal_self_attention): MultiheadAttention(
        (w_q): Linear(in_features=512, out_features=512, bias=True)
        (w_k): Linear(in_features=512, out_features=512, bias=True)
        (w_v): Linear(in_features=512, out_features=512, bias=True)
        (w_o): Linear(in_features=512, out_features=512, bias=True)
        (attn_dropout): Dropout(p=0.1, inplace=False)
      )
      (cross_attention): MultiheadAttention(
        (w_q): Linear(in_features=512, out_features=512, bias=True)
        (w_k): Linear(in_features=512, out_features=512, bias=True)
        (w_v): Linear(in_features=512, out_features=512, bias=True)
        (w_o): Linear(in_features=512, out_features=512, bias=True)
        (attn_dropout): Dropout(p=0.1, inplace=False)
      )
      (ffn): Sequential(
        (0): Linear(in_features=512, out_features=2048, bias=True)
        (1): ReLU()
        (2): Linear(in_features=2048, out_features=

## 5. Complete Transformer

The model uses one token embedding matrix for the source and target. The language-model head is tied to that same matrix, matching the weight-sharing scheme described in the original Transformer.

The final hidden state at every decoder position is projected to one logit per vocabulary token. Cross-entropy compares those logits with the shifted labels, ignores padded label positions, and uses the original paper's label-smoothing value of $\epsilon_{ls}=0.1$.

In [9]:
class Transformer(nn.Module):
    def __init__(
            self,
            vocab_size,
            embedding_dim,
            num_heads,
            context_length,
            num_layers,
            pad_token_id,
            dropout_rate=0.1,
            label_smoothing=0.1,
    ):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.pad_token_id = pad_token_id
        self.label_smoothing = label_smoothing
        self.sqrt_d_model = embedding_dim ** 0.5

        # input embeddings
        self.token_embeddings = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_token_id
        )
        self.position_encodings = SinusoidalPositionEmbedding(
            context_length=context_length,
            embedding_dim=embedding_dim
        )
        self.embedding_dropout = nn.Dropout(dropout_rate)

        # stacked encoder
        self.encoder = TransformerEncoder(
            embedding_dim=embedding_dim,
            num_heads=num_heads,
            context_length=context_length,
            num_layers=num_layers,
            dropout_rate=dropout_rate
        )

        # stacked decoder
        self.decoder = TransformerDecoder(
            embedding_dim=embedding_dim,
            num_heads=num_heads,
            context_length=context_length,
            num_layers=num_layers,
            dropout_rate=dropout_rate
        )

        # output
        self.lm_head = nn.Linear(embedding_dim, vocab_size, bias=False)

        # initialize the weights
        self.apply(self._init_weights)
        self.lm_head.weight = self.token_embeddings.weight
        with torch.no_grad():
            self.token_embeddings.weight[pad_token_id].zero_()

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=module.embedding_dim**-0.5)

    def _embed(self, token_ids: torch.Tensor):
        seq_len = token_ids.size(1)
        tok_embeds = self.token_embeddings(token_ids)
        positions = torch.arange(seq_len, device=token_ids.device)
        pos_embeds = self.position_encodings(positions)

        return self.embedding_dropout(
            tok_embeds * self.sqrt_d_model + pos_embeds
        )

    def forward(
            self,
            source_ids: torch.Tensor,
            decoder_input_ids: torch.Tensor,
            source_padding_mask: torch.Tensor,
            decoder_padding_mask: torch.Tensor,
            labels: torch.Tensor = None
    ):
        if source_padding_mask is None:
            source_padding_mask = source_ids.eq(self.pad_token_id)
        if decoder_padding_mask is None:
            decoder_padding_mask = decoder_input_ids.eq(self.pad_token_id)

        encoder_input = self._embed(source_ids)
        decoder_input = self._embed(decoder_input_ids)

        encoder_hidden_states = self.encoder(encoder_input, padding_mask=source_padding_mask)
        decoder_hidden_states = self.decoder(
            decoder_input,
            encoder_hidden_states=encoder_hidden_states,
            decoder_padding_mask=decoder_padding_mask,
            encoder_padding_mask=source_padding_mask
        )

        logits = self.lm_head(decoder_hidden_states)
        # logits shape:  [B, L, vocab_size]
        # current labels shaoe:  [B, L]

        loss = None
        if labels is not None:
            # current logits shape:  [B, L, vocab_size]
            # current labels shaoe:  [B, L]
            # need to reshape logits & labels for computing the loss
            # logits → [B * L, vocab_size]
            # labels → [B * L]
            loss = F.cross_entropy(
                logits.reshape(-1, logits.shape[-1]),
                labels.reshape(-1),
                ignore_index=self.pad_token_id,
                label_smoothing=self.label_smoothing
            )

        return {"logits": logits, "loss": loss}
    
# testing
transformer = Transformer(1000, 128, 8, 128, 6, 0, 0.1, 0.1)
transformer

Transformer(
  (token_embeddings): Embedding(1000, 128, padding_idx=0)
  (position_encodings): SinusoidalPositionEmbedding()
  (embedding_dropout): Dropout(p=0.1, inplace=False)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (self_attention): MultiheadAttention(
          (w_q): Linear(in_features=128, out_features=128, bias=True)
          (w_k): Linear(in_features=128, out_features=128, bias=True)
          (w_v): Linear(in_features=128, out_features=128, bias=True)
          (w_o): Linear(in_features=128, out_features=128, bias=True)
          (attn_dropout): Dropout(p=0.1, inplace=False)
        )
        (ffn): Sequential(
          (0): Linear(in_features=128, out_features=512, bias=True)
          (1): ReLU()
          (2): Linear(in_features=512, out_features=128, bias=True)
        )
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (ffn_dropout): Dropout(p=0.1, inplace=False)
        (layer_norm1): LayerNorm((128,

## 6. End-to-end integration check

First, let's do an integration check for the complete architecture rather than a training loop. This verifies that data preparation, the stacked encoder-decoder, tied vocabulary projection, loss, and backpropagation work together.

In [10]:
model = Transformer(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    context_length=context_length,
    pad_token_id=pad_token_id,
    dropout_rate=dropout_rate,
    label_smoothing=label_smoothing,
)

outputs = model(**batch)
loss = outputs["loss"]
loss.backward()
model.zero_grad(set_to_none=True)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Logits shape: {tuple(outputs['logits'].shape)}")
print(f"Loss: {loss.item():.4f}")
print(f"Embedding/output weights tied: {model.lm_head.weight is model.token_embeddings.weight}")

Model parameters: 52,330,496
Logits shape: (64, 25, 16000)
Loss: 10.2233
Embedding/output weights tied: True


## 7. Full single-GPU training

This recipe retains the original Transformer-base architecture and its Adam settings ($\beta_1=0.9$, $\beta_2=0.98$, $\epsilon=10^{-9}$), inverse-square-root schedule with 4,000 warm-up updates, dropout 0.1, and label smoothing 0.1.

The paper used eight GPUs and approximately 25,000 source plus 25,000 target tokens per update. On one 24 GB RTX 3090 Ti, a BF16 micro-batch of 64 sentence pairs with four-step accumulation gives an effective batch of 256 pairs. Dynamic padding keeps this practical, and the loop reports actual non-padding tokens. It stops after five wall-clock hours by default; set `max_training_hours` between 4 and 6. Checkpoints support deterministic mid-epoch resume.


In [11]:
seed = 42
validation_examples = 5_000
micro_batch_size = 64
gradient_accumulation_steps = 4
num_workers = 4
warmup_steps = 4_000
max_training_hours = 8.0
validation_interval = 500
checkpoint_interval = 1_000
gradient_clip_norm = 1.0
log_interval = 20

checkpoint_dir = repo_root / "checkpoints" / "transformer_en_de"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
latest_checkpoint_path = checkpoint_dir / "latest.pt"
best_checkpoint_path = checkpoint_dir / "best.pt"

torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
if not torch.cuda.is_available():
    raise RuntimeError("This training recipe expects a CUDA GPU.")
device = torch.device("cuda")
use_bf16 = torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

train_size = len(all_dataset) - validation_examples
if train_size <= 0:
    raise ValueError("validation_examples must be smaller than the dataset")
train_dataset, validation_dataset = random_split(
    all_dataset, [train_size, validation_examples],
    generator=torch.Generator().manual_seed(seed),
)
collator = TranslationCollator(pad_token_id)

def make_train_loader(epoch):
    return DataLoader(
        train_dataset, batch_size=micro_batch_size, shuffle=True,
        generator=torch.Generator().manual_seed(seed + epoch),
        collate_fn=collator, num_workers=num_workers, pin_memory=True,
        persistent_workers=num_workers > 0, drop_last=True,
    )

validation_loader = DataLoader(
    validation_dataset, batch_size=micro_batch_size, shuffle=False,
    collate_fn=collator, num_workers=num_workers, pin_memory=True,
    persistent_workers=num_workers > 0,
)
print(f"Device: {torch.cuda.get_device_name(0)}; precision: {amp_dtype}")
print(f"Train/validation: {len(train_dataset):,}/{len(validation_dataset):,}")
print(f"Effective batch: {micro_batch_size * gradient_accumulation_steps} pairs")


Device: NVIDIA GeForce RTX 3090 Ti; precision: torch.bfloat16
Train/validation: 317,406/5,000
Effective batch: 256 pairs


### Original learning-rate schedule

At optimizer update $s$, the paper uses $d_{model}^{-0.5}\min(s^{-0.5}, s\,warmup^{-1.5})$. The rate rises linearly for 4,000 updates and then decays with the inverse square root of the update number.


In [12]:
def transformer_learning_rate(step, d_model=embedding_dim, warmup=warmup_steps):
    step = max(step, 1)
    return d_model**-0.5 * min(step**-0.5, step * warmup**-1.5)


def move_batch_to_device(batch):
    return {name: tensor.to(device, non_blocking=True) for name, tensor in batch.items()}


@torch.inference_mode()
def evaluate(model, data_loader):
    model.eval()
    loss_sum = token_count = 0
    for batch in data_loader:
        batch = move_batch_to_device(batch)
        tokens = batch["labels"].ne(pad_token_id).sum().item()
        with torch.autocast("cuda", dtype=amp_dtype):
            loss = model(**batch)["loss"]
        loss_sum += loss.item() * tokens
        token_count += tokens
    model.train()
    return loss_sum / max(token_count, 1)


def save_checkpoint(path, model, optimizer, epoch, next_batch, update, best_loss, elapsed):
    torch.save({
        "model": model.state_dict(), "optimizer": optimizer.state_dict(),
        "epoch": epoch, "next_batch": next_batch, "update": update,
        "best_validation_loss": best_loss, "elapsed_seconds": elapsed,
        "config": {
            "vocab_size": vocab_size, "embedding_dim": embedding_dim,
            "num_heads": num_heads, "num_layers": num_layers,
            "context_length": context_length, "pad_token_id": pad_token_id,
            "dropout_rate": dropout_rate, "label_smoothing": label_smoothing,
        },
    }, path)


In [ ]:
resume_training = False

model = Transformer(
    vocab_size, embedding_dim, num_heads, context_length, num_layers,
    pad_token_id, dropout_rate, label_smoothing,
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0, betas=(0.9, 0.98), eps=1e-9)
scaler = torch.amp.GradScaler("cuda", enabled=not use_bf16)
epoch = next_batch = update = 0
best_validation_loss = math.inf
previous_elapsed = 0.0

if resume_training:
    checkpoint = torch.load(latest_checkpoint_path, map_location="cpu", weights_only=False)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    epoch, next_batch, update = checkpoint["epoch"], checkpoint["next_batch"], checkpoint["update"]
    best_validation_loss = checkpoint["best_validation_loss"]
    previous_elapsed = checkpoint["elapsed_seconds"]

started = time.monotonic()
time_budget = max_training_hours * 3600
model.train()
optimizer.zero_grad(set_to_none=True)
stop = False
last_batch = next_batch
accumulation_count = 0

while not stop:
    loader = make_train_loader(epoch)
    progress = tqdm(enumerate(loader), total=len(loader), desc=f"epoch {epoch + 1}")
    token_count = 0
    for batch_index, batch in progress:
        if batch_index < next_batch:
            continue
        last_batch = batch_index
        batch = move_batch_to_device(batch)
        token_count += (
            batch["source_ids"].ne(pad_token_id).sum()
            + batch["labels"].ne(pad_token_id).sum()
        ).item()
        with torch.autocast("cuda", dtype=amp_dtype):
            loss = model(**batch)["loss"] / gradient_accumulation_steps
        scaler.scale(loss).backward()
        accumulation_count += 1
        if accumulation_count < gradient_accumulation_steps:
            continue
        accumulation_count = 0

        update += 1
        lr = transformer_learning_rate(update)
        for group in optimizer.param_groups:
            group["lr"] = lr
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip_norm)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        elapsed = previous_elapsed + time.monotonic() - started

        if update % log_interval == 0:
            progress.set_postfix(loss=f"{loss.item() * gradient_accumulation_steps:.3f}", lr=f"{lr:.2e}", tokens=f"{token_count:,}")
        token_count = 0

        if update % validation_interval == 0:
            val_loss = evaluate(model, validation_loader)
            print(f"update {update:,} | val loss {val_loss:.4f} | {elapsed / 3600:.2f} h")
            if val_loss < best_validation_loss:
                best_validation_loss = val_loss
                save_checkpoint(best_checkpoint_path, model, optimizer, epoch, batch_index + 1, update, best_validation_loss, elapsed)
        if update % checkpoint_interval == 0:
            save_checkpoint(latest_checkpoint_path, model, optimizer, epoch, batch_index + 1, update, best_validation_loss, elapsed)
        if elapsed >= time_budget:
            stop = True
            break
    if not stop:
        epoch += 1
        next_batch = 0

elapsed = previous_elapsed + time.monotonic() - started
save_checkpoint(latest_checkpoint_path, model, optimizer, epoch, last_batch + 1, update, best_validation_loss, elapsed)
print(f"Finished after {elapsed / 3600:.2f} h, {update:,} updates; validation loss {evaluate(model, validation_loader):.4f}")


epoch 1:   0%|          | 0/4959 [00:00<?, ?it/s]

update 500 | val loss 5.6442 | 0.02 h
update 1,000 | val loss 4.8338 | 0.03 h


epoch 2:   0%|          | 0/4959 [00:00<?, ?it/s]

update 1,500 | val loss 4.3800 | 0.05 h
update 2,000 | val loss 4.0301 | 0.07 h


epoch 3:   0%|          | 0/4959 [00:00<?, ?it/s]

update 2,500 | val loss 3.7755 | 0.08 h
update 3,000 | val loss 3.6320 | 0.10 h
update 3,500 | val loss 3.5293 | 0.12 h


... (59 outputs left out)

update 40,000 | val loss 2.7953 | 1.47 h
update 40,500 | val loss 2.7975 | 1.49 h


epoch 34:   0%|          | 0/4959 [00:00<?, ?it/s]

update 41,000 | val loss 2.7974 | 1.51 h
update 41,500 | val loss 2.7992 | 1.52 h
update 42,000 | val loss 2.7832 | 1.54 h


epoch 35:   0%|          | 0/4959 [00:00<?, ?it/s]

## 8. Export and upload to the Hugging Face Hub

Authenticate outside the notebook with `hf auth login`, or set `HF_TOKEN`; never paste a token into a cell. The authenticated username is discovered automatically. The export includes Safetensors weights, tokenizer, configuration, and a model card. Because this is a plain PyTorch module rather than a `transformers.PreTrainedModel`, consumers need this notebook's class definitions to load it.


In [ ]:
run_hub_upload = True
hub_model_name = "transformer-en-de-tatoeba"
private_repo = False

if run_hub_upload:
    api = HfApi()
    username = api.whoami()["name"]
    repo_id = f"{username}/{hub_model_name}"
    export_dir = checkpoint_dir / "hub_export"
    export_dir.mkdir(parents=True, exist_ok=True)
    source_checkpoint = best_checkpoint_path if best_checkpoint_path.exists() else latest_checkpoint_path
    checkpoint = torch.load(source_checkpoint, map_location="cpu", weights_only=False)
    model.load_state_dict(checkpoint["model"])
    model.cpu().eval()
    save_model(model, export_dir / "model.safetensors")
    shutil.copy2(tokenizer_path, export_dir / "tokenizer.json")
    config = {
        **checkpoint["config"], "architecture": "Transformer",
        "source_language": "en", "target_language": "de",
        "bos_token_id": bos_token_id, "eos_token_id": eos_token_id,
        "training_updates": checkpoint["update"],
        "best_validation_loss": checkpoint["best_validation_loss"],
    }
    (export_dir / "config.json").write_text(json.dumps(config, indent=2) + "\n")
    card = f"""---
language: [en, de]
pipeline_tag: translation
library_name: pytorch
tags: [transformer, translation, from-scratch]
license: apache-2.0
---
# Transformer English-German (Tatoeba)

A from-scratch {num_layers}-layer Transformer encoder-decoder with hidden size
{embedding_dim}, {num_heads} attention heads, and a shared {vocab_size:,}-token BPE vocabulary.
Load it with the class definitions in `notebooks/00_transformer_components/4_transformer.ipynb`.
"""
    (export_dir / "README.md").write_text(card)
    api.create_repo(repo_id, private=private_repo, exist_ok=True)
    api.upload_folder(repo_id=repo_id, folder_path=export_dir, commit_message="Upload trained Transformer")
    print(f"Uploaded to https://huggingface.co/{repo_id}")
else:
    print("Upload disabled. Set run_hub_upload = True after training.")
